In [2]:
import neurokit2 as nk
import mne
import os
import os.path as op
import numpy as np
import pandas as  pd
import matplotlib.pyplot as plt
import re
import numpy
from statsmodels.stats.anova import AnovaRM
from mne.time_frequency import tfr_morlet


In [ ]:
subjects = ['AS32', 'CB55', 'DE21', 'DF23', 'EE89', 'FK11', 'FK24', 'fx48',
       'GF87', 'GG88', 'GS21', 'HG11', 'JK67', 'JU77', 'KI09', 'KL72',
       'KO33', 'LL90', 'MN99', 'MO45', 'NO21', 'OO11', 'OP01', 'PO67',
       'PP00', 'RD56', 'RO88', 'RT12', 'TR90', 'TY65', 'YT50']

**EXTRACTING QUESTIONNAIRE RESULTS**

In [60]:
for subject in subjects:
    data = []

    # Open the file in read mode ('r')
    with open(f'D:/CYBERDELICS/RAW FILES/{subject}/{subject}.txt', 'r') as file:
    # Read the file line by line
        for line in file:
            # Use regular expression to extract the condition and number
            match = re.match(r'^([a-zA-Z]+): (\d+)$', line)
            if match:
                # If a match is found, extract filename, condition, and number
                subj = subject  # Replace with the actual filename
                label = match.group(1)
                number = int(match.group(2))

                # Add data to the list
                data.append([subject, label, number])

    # Create a DataFrame from the list of data
    df = pd.DataFrame(data, columns=['Subject', 'Condition', 'Number'])
    #df.to_csv(f'D:/hse/psychodelic_like_experience/data_processing/otvety/{subject}.csv', index=False)
    df.to_csv(f'...path/{subject}.csv', index=False)

    # Now, 'df' contains the desired DataFrame
    print(df)


  Subject                Condition  Number
0    JK67              FractalBlue       1
1    JK67     HoneyCombGreenPurple       4
2    JK67  CubesControlGreenOrange       4
3    JK67         kaleidoscopeBlue       0
4    JK67      CubesControlBlueRed       3
5    JK67             FractalGreen       1
6    JK67        HoneyCombBluePink       2
7    JK67          kaleidoscopeRed       0
  Subject               Condition  Number
0    TY65             FractalBlue       0
1    TY65    CubesControlPinkBlue       2
2    TY65       kaleidoscopeGreen       0
3    TY65       HoneyCombBluePink       2
4    TY65      kaleidoscopeYellow       0
5    TY65  CubesControlPinkYellow       1
6    TY65        HoneyCombRedBlue       2
7    TY65            FractalGreen       1
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none


**EVENTS EXTRACTION**

In [176]:
#dictionary with codes
all_events_id = {
    'FractalBlue': 101,
    'FractalPurple': 102,
    'FractalGreen': 103,
    'FractalRed': 104,
    'FractalYellow': 105,

    'kaleidoscopeBlue': 201,
    'kaleidoscopePurple': 202,
    'kaleidoscopeGreen': 203,
    'kaleidoscopeRed': 204,
    'kaleidoscopeYellow': 205,

    'CubesControlPinkBlue': 301,
    'CubesControlPurpleGreen': 302,
    'CubesControlBlueRed': 303,
    'CubesControlPinkYellow': 304,
    'CubesControlGreenOrange': 305,

    'HoneyCombBluePink': 401,
    'HoneyCombGreenPurple': 402,
    'HoneyCombRedBlue': 403,
    'HoneyCombOrangePink': 404,
    'HoneyCombGreenOrange': 405
}
event_id = {}

for key, value in all_events_id.items():
    new_key = key.replace("HoneyComb", "HoneyComb/")\
                  .replace("kaleidoscope", "kaleidoscope/")\
                  .replace("Fractal", "Fractal/")\
                  .replace("CubesControl", "CubesControl/")

    event_id[new_key] = value

print(event_id)

{'Fractal/Blue': 101, 'Fractal/Purple': 102, 'Fractal/Green': 103, 'Fractal/Red': 104, 'Fractal/Yellow': 105, 'kaleidoscope/Blue': 201, 'kaleidoscope/Purple': 202, 'kaleidoscope/Green': 203, 'kaleidoscope/Red': 204, 'kaleidoscope/Yellow': 205, 'CubesControl/PinkBlue': 301, 'CubesControl/PurpleGreen': 302, 'CubesControl/BlueRed': 303, 'CubesControl/PinkYellow': 304, 'CubesControl/GreenOrange': 305, 'HoneyComb/BluePink': 401, 'HoneyComb/GreenPurple': 402, 'HoneyComb/RedBlue': 403, 'HoneyComb/OrangePink': 404, 'HoneyComb/GreenOrange': 405}


In [56]:
for subject in subjects:
    conditions = []

    # Open the file in read mode ('r')
    with open(f'D:/CYBERDELICS/RAW FILES/{subject}/{subject}.txt', 'r') as file:
        # Read the file line by line
        for line in file:
            # Use regular expression to extract the condition
            match = re.match(r'^([a-zA-Z]+)', line)
            if match:
                # If a match is found, add the label to the list
                conditions.append(match.group(1))

    # Now, 'conditions' contains the extracted conditions
    print(conditions)
    labels = [all_events_id.get(condition, None) for condition in conditions]

    # Print the list of labels
    print(labels)
    raw = mne.io.read_raw_brainvision(f'D:/hse/psychodelic_like_experience/отдельный разговор/{subject}/{subject}.vhdr', preload = True)
    mark_array = raw.pick('mark').get_data().flatten()
    events = nk.events_find(event_channel = mark_array, threshold='auto', threshold_keep='below', start_at=0, end_at=None, duration_min=0, duration_max=None, inter_min=0, discard_first=2, discard_last=1, event_labels=labels, event_conditions=conditions)
    events
    events_mne = np.column_stack((events['onset'], np.zeros_like(events['onset']), events['label']))
    events_mne
    #mne.write_events(f'D:/hse/psychodelic_like_experience/data_processing/events/{subject}_eve.txt', events_mne, overwrite=True, verbose=None)
    mne.write_events(f'...path/events/{subject}_eve.txt', events_mne, overwrite=True, verbose=None)
    #nk.events_plot(events, mark_array)
    

['FractalBlue', 'CubesControlPinkBlue', 'kaleidoscopeGreen', 'HoneyCombBluePink', 'kaleidoscopeRed', 'CubesControlPurpleGreen', 'kaleidoscopePurple', 'HoneyCombGreenPurple', 'CubesControlBlueRed', 'kaleidoscopeBlue', 'CubesControlGreenOrange', 'HoneyCombGreenOrange', 'kaleidoscopeYellow', 'HoneyCombOrangePink', 'CubesControlPinkYellow', 'FractalYellow', 'HoneyCombRedBlue', 'FractalRed', 'FractalPurple', 'FractalGreen']
[101, 301, 203, 401, 204, 302, 202, 402, 303, 201, 305, 405, 205, 404, 304, 105, 403, 104, 102, 103]
Extracting parameters from D:/hse/psychodelic_like_experience/отдельный разговор/TY65/TY65.vhdr...
Setting channel info structure...
Reading 0 ... 797324  =      0.000 ...  1594.648 secs...
Overwriting existing file.


**EPOCHS CREATION**

In [177]:
epochs_all = []
for subject in subjects:
    
    raw = mne.io.read_raw(f'D:/CYBERDELICS/CLEANED RAW/{subject}cleaned_raw.fif', preload = True)
    raw.plot(block = True)
    raw = raw.copy().interpolate_bads(reset_bads=False)
    events = mne.read_events(f'D:/hse/psychodelic_like_experience/data_processing/events/{subject}_eve.txt')
    epochs = mne.Epochs(raw, events, event_id, tmin=-5, tmax=60, baseline=None, preload = True)
    epochs.plot( n_epochs=1, n_channels=20, title=None, show=True, decim='auto', block = True)
    #for epoch in epochs:
    epochs.save(f'...path/{subject}_epo.fif', overwrite=True)
    
    epochs_all.append(epochs)

Opening raw data file D:/hse/psychodelic_like_experience/data_processing/eeg_cleaned_ica/AS32cleaned_raw.fif...
    Read a total of 2 projection items:
        EOG-eeg--0.200-0.200-PCA-01 (1 x 28)  idle
        EOG-eeg--0.200-0.200-PCA-02 (1 x 28)  idle
    Range : 0 ... 797474 =      0.000 ...  1594.948 secs
Ready.
Reading 0 ... 797474  =      0.000 ...  1594.948 secs...
Channels marked as bad:
['P8', 'FC2']
Interpolating bad channels
    Automatic origin fit: head of radius 94.6 mm
Computing interpolation matrix from 26 sensor positions
Interpolating 2 sensors
Not setting metadata
20 matching events found
No baseline correction applied
Created an SSP operator (subspace dimension = 2)
2 projection items activated
Using data from preloaded Raw for 20 events and 32501 original time points ...
0 bad epochs dropped


C:\Users\User\AppData\Local\Temp\ipykernel_1256\3130389144.py:9: FutureWarning: The current default events=None is deprecated and will change to events=True in MNE 1.6. Set events=False to suppress this warning.
  epochs.plot( n_epochs=1, n_channels=20, title=None, show=True, decim='auto', block = True)


Dropped 15 epochs: 1, 2, 5, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19
The following epochs were marked as bad and are dropped:
[1, 2, 5, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Channels marked as bad:
['P8', 'FC2']
Opening raw data file D:/hse/psychodelic_like_experience/data_processing/eeg_cleaned_ica/CB55cleaned_raw.fif...
    Read a total of 2 projection items:
        EOG-eeg--0.200-0.200-PCA-01 (1 x 28)  idle
        EOG-eeg--0.200-0.200-PCA-02 (1 x 28)  idle
    Range : 0 ... 821549 =      0.000 ...  1643.098 secs
Ready.
Reading 0 ... 821549  =      0.000 ...  1643.098 secs...
Channels marked as bad:
['O1', 'CP1']
Interpolating bad channels
    Automatic origin fit: head of radius 94.6 mm
Computing interpolation matrix from 26 sensor positions
Interpolating 2 sensors
Not setting metadata
20 matching events found
No baseline correction applied
Created an SSP operator (subspace dimension = 2)
2 projection items activated
Using data from preloaded Raw for 20 events and 3250

C:\Users\User\AppData\Local\Temp\ipykernel_1256\3130389144.py:9: FutureWarning: The current default events=None is deprecated and will change to events=True in MNE 1.6. Set events=False to suppress this warning.
  epochs.plot( n_epochs=1, n_channels=20, title=None, show=True, decim='auto', block = True)


Dropped 4 epochs: 0, 2, 5, 17
The following epochs were marked as bad and are dropped:
[0, 2, 5, 17]
Channels marked as bad:
['O1', 'CP1']
Opening raw data file D:/hse/psychodelic_like_experience/data_processing/eeg_cleaned_ica/DE21cleaned_raw.fif...
    Read a total of 2 projection items:
        EOG-eeg--0.200-0.200-PCA-01 (1 x 28)  idle
        EOG-eeg--0.200-0.200-PCA-02 (1 x 28)  idle
    Range : 0 ... 802799 =      0.000 ...  1605.598 secs
Ready.
Reading 0 ... 802799  =      0.000 ...  1605.598 secs...


...

Dropped 0 epochs: 
The following epochs were marked as bad and are dropped:
[]
Channels marked as bad:
['CP1']
Opening raw data file D:/hse/psychodelic_like_experience/data_processing/eeg_cleaned_ica/YT50cleaned_raw.fif...
    Read a total of 2 projection items:
        EOG-eeg--0.200-0.200-PCA-01 (1 x 28)  idle
        EOG-eeg--0.200-0.200-PCA-02 (1 x 28)  idle
    Range : 0 ... 802449 =      0.000 ...  1604.898 secs
Ready.
Reading 0 ... 802449  =      0.000 ...  1604.898 secs...
Channels marked as bad:
none
Not setting metadata
20 matching events found
No baseline correction applied
Created an SSP operator (subspace dimension = 2)
2 projection items activated
Using data from preloaded Raw for 20 events and 32501 original time points ...


C:\Users\User\AppData\Local\Temp\ipykernel_1256\3130389144.py:6: RuntimeWarning: No bad channels to interpolate. Doing nothing...
  raw = raw.copy().interpolate_bads(reset_bads=False)


0 bad epochs dropped


C:\Users\User\AppData\Local\Temp\ipykernel_1256\3130389144.py:9: FutureWarning: The current default events=None is deprecated and will change to events=True in MNE 1.6. Set events=False to suppress this warning.
  epochs.plot( n_epochs=1, n_channels=20, title=None, show=True, decim='auto', block = True)


Dropped 11 epochs: 1, 2, 4, 5, 9, 11, 12, 13, 15, 18, 19
The following epochs were marked as bad and are dropped:
[1, 2, 4, 5, 9, 11, 12, 13, 15, 18, 19]
Channels marked as bad:
none
